## Lesson 3: Email Assistant with Semantic Memory


In [ ]:
# 加载 .env 里的环境变量（OPENAI_API_KEY / ANTHROPIC_API_KEY 等）
import os
from dotenv import load_dotenv
_ = load_dotenv()

## Repeat setup from previous lesson

In [ ]:
# 用户画像，和 lesson_2 完全一致，本课复用
profile = {
    "name": "John",
    "full_name": "John Doe",
    "user_profile_background": "Senior software engineer leading a team of 5 developers",
}

In [ ]:
# 分诊规则 + agent 行为指令，同 lesson_2
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage John's tasks efficiently."
}

In [ ]:
# 示例邮件，同 lesson_2
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "body": """
Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

In [ ]:
# BaseModel/Field：定义结构化输出 schema；TypedDict：定义 State；init_chat_model：统一模型加载入口
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

In [ ]:
# 分诊用的小模型
llm = init_chat_model("openai:gpt-4o-mini")

In [ ]:
# Router：分诊分类的结构化输出 schema，同 lesson_2
class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

In [ ]:
# 包装成结构化输出模型
llm_router = llm.with_structured_output(Router)

In [ ]:
# 复用 notebook 1 里补齐的 prompts.py（该目录原本缺失此文件，详见 notebook 1 里的说明）
from prompts import triage_system_prompt, triage_user_prompt

In [ ]:
# @tool 装饰器：把普通函数注册为 LangChain 工具
from langchain_core.tools import tool

In [ ]:
# 工具 1：发送邮件（占位实现）
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    # Placeholder response - in real app would send email
    return f"Email sent to {to} with subject '{subject}'"


In [ ]:
# 工具 2：安排会议（占位实现）
@tool
def schedule_meeting(
    attendees: list[str],
    subject: str,
    duration_minutes: int,
    preferred_day: str
) -> str:
    """Schedule a calendar meeting."""
    # Placeholder response - in real app would check calendar and schedule
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


In [ ]:
# 工具 3：查看日历可用时段（占位实现）
@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # Placeholder response - in real app would check actual calendar
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

## Define tools for managing memory

**架构说明**：这一课引入 LangGraph 的**长期记忆（long-term memory）**能力。
不同于 `messages` 那种只在一次会话内存在的短期状态，长期记忆通过 `Store`
持久化存储，可以跨会话、跨线程被检索——这样 agent 才能记住"Jim 是用户的朋友"
这类事实，下次被问起时还能答上来。

In [ ]:
# InMemoryStore：LangGraph 提供的长期记忆存储实现（进程内存，重启即丢失；
# 生产环境可换成 Postgres/Redis 等持久化 Store，接口一致）
from langgraph.store.memory import InMemoryStore

In [ ]:
# index={"embed": ...}：给 store 配置向量索引，写入的记忆会自动生成 embedding，
# 之后就能用 store.search(query=...) 做语义检索（这就是"语义记忆 Semantic Memory"名字的由来）
store = InMemoryStore(
    index={"embed": "openai:text-embedding-3-small"}
)

In [ ]:
# langmem：LangChain 官方的记忆管理工具库，提供开箱即用的"记忆读写工具"，
# 不用自己手写 store.put/store.search 的调用逻辑
# 注意：这份目录原本没有装 langmem（pip 安装清单里没有它），已额外 pip install langmem 补上
from langmem import create_manage_memory_tool, create_search_memory_tool

In [ ]:
# create_manage_memory_tool / create_search_memory_tool：
# 分别生成"写入/更新/删除记忆"和"检索记忆"两个 LangChain 工具，交给 agent 自主调用。
#
# namespace 里的 "{langgraph_user_id}" 是一个占位符：真正调用时会被
# config["configurable"]["langgraph_user_id"] 的值替换，这样不同用户的记忆
# 天然被隔离在不同 namespace 下，不会互相串用。
#
# TODO: 请在此处补全代码
# 用 create_manage_memory_tool(namespace=(...)) 创建 manage_memory_tool
# 用 create_search_memory_tool(namespace=(...)) 创建 search_memory_tool
# namespace 都是 ("email_assistant", "{langgraph_user_id}", "collection")
manage_memory_tool = None
search_memory_tool = None

In [ ]:
# 看看 langmem 自动生成的工具名（会作为工具调用时的函数名暴露给 LLM）
print(manage_memory_tool.name)

In [ ]:
# 工具描述（LLM 靠这个判断什么时候该调用它）
print(manage_memory_tool.description)

In [ ]:
# 查看工具的参数 schema：content/action(create|update|delete)/id
manage_memory_tool.args

In [ ]:
search_memory_tool.name

In [ ]:
search_memory_tool.description

In [ ]:
# 查询工具的参数 schema：query/limit/offset/filter
search_memory_tool.args

In [ ]:
# 主 agent 的系统提示词：比 lesson_2 的版本多了工具 4、5（manage_memory / search_memory）
# 显式告诉 LLM 有记忆能力可用，否则它不知道该在什么时候调用这两个工具
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:

1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc. in memory for future reference
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""

In [ ]:
# TODO: 请在此处补全代码
# 实现 create_prompt(state)：
#   1. 用 agent_system_prompt_memory.format(instructions=..., **profile) 生成系统提示词
#   2. 返回 [{"role": "system", "content": ...}] + state['messages']
def create_prompt(state):
    pass

In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
# 工具列表比 lesson_2 多了 manage_memory_tool / search_memory_tool 两个记忆工具
tools= [
    write_email,
    schedule_meeting,
    check_calendar_availability,
    manage_memory_tool,
    search_memory_tool
]
# TODO: 请在此处补全代码
# 用 create_react_agent(model, tools=tools, prompt=create_prompt, store=store) 构建 response_agent
# 注意一定要传 store=store，否则 manage_memory_tool/search_memory_tool 拿不到 store 实例
response_agent = None

In [ ]:
# langgraph_user_id="lance"：决定了 manage_memory_tool/search_memory_tool 里
# "{langgraph_user_id}" 占位符最终解析成哪个用户的记忆命名空间
config = {"configurable": {"langgraph_user_id": "lance"}}

In [ ]:
# 告诉 agent "Jim is my friend"，预期 LLM 会主动调用 manage_memory 工具把这条信息存进 store
# 真实网络请求，假 key 场景下预期报鉴权错误
response = response_agent.invoke(
    {"messages": [{"role": "user", "content": "Jim is my friend"}]},
    config=config
)

In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
# 再问"who is jim?"，预期 agent 这次会调用 search_memory 工具，从 store 里把刚存的记忆检索出来
response = response_agent.invoke(
    {"messages": [{"role": "user", "content": "who is jim?"}]},
    config=config
)

In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
# 直接绕开 agent，查看 store 里实际存了哪些命名空间（不需要联网，纯本地数据结构查询）
store.list_namespaces()

In [ ]:
# 不带 query 的 search：相当于列出该命名空间下的全部记忆条目（不需要向量检索，不联网）
store.search(('email_assistant', 'lance', 'collection'))

In [ ]:
# 带 query="jim" 的语义检索：这一步需要先把 "jim" 转成 embedding 再做相似度匹配，
# 所以会真正调用 OpenAI 的 embedding API（假 key 场景下预期报错），
# 和上面不带 query 的纯本地查询不同
store.search(('email_assistant', 'lance', 'collection'), query="jim")

## Create the rest of the agent

In [ ]:
from langgraph.graph import add_messages

# TODO: 请在此处补全代码
# 定义 State(TypedDict)，字段: email_input: dict, messages: Annotated[list, add_messages]
class State(TypedDict):
    pass

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

In [ ]:
# TODO: 请在此处补全代码
# 实现 triage_router(state) -> Command[Literal["response_agent","__end__"]]
# 步骤同 notebook 1 里的 triage_router：
#   1. 取出 author/to/subject/email_thread
#   2. 拼 system_prompt / user_prompt
#   3. llm_router.invoke(...) 得到 result
#   4. 按 result.classification 分三种情况设置 goto / update
#   5. 最后 return Command(goto=goto, update=update)（别漏了这个 return！）
def triage_router(state: State) -> Command[
    Literal["response_agent", "__end__"]
]:
    pass

## Create email agent


In [ ]:
# TODO: 请在此处补全代码
# 用 StateGraph(State) 组装图：
#   add_node(triage_router)
#   add_node("response_agent", response_agent)
#   add_edge(START, "triage_router")
#   compile(store=store)   # 别忘了传 store，这样图内节点才能访问长期记忆
email_agent = None

In [ ]:
# 可视化图结构（需要联网访问 mermaid.ink，与 API key 无关）
display(Image(email_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# 测试邮件：预期分类为 respond，进入 response_agent
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

In [ ]:
# 用 config 传入 langgraph_user_id="lance"，这样 response_agent 里的记忆工具
# 会读写到和上面测试同一个用户命名空间下
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

In [ ]:
for m in response["messages"]:
    m.pretty_print()

## Try a follow-up email

**架构说明**：这里故意发一封信息很少的"追问"邮件（"有更新吗？"），
用来验证 agent 是否真的能借助长期记忆（而不是短期的 messages 历史）
理解"previous ask"指的是什么——因为这是一次全新的 `invoke` 调用，
之前那次对话的 `messages` 并不会自动带过来，能接得上全靠 search_memory。

In [ ]:
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Follow up",
    "email_thread": """Hi John,

Any update on my previous ask?""",
}

In [ ]:
response = email_agent.invoke({"email_input": email_input}, config=config)

In [ ]:
for m in response["messages"]:
    m.pretty_print()